# Running Models — Audio · Part 6 — Companion Notebook

This goes with **"Type a Vibe, Get a Track: Text-to-Music with MusicGen"**. Run cells top to bottom (Shift+Enter).

**Use a GPU:** Runtime → Change runtime type → GPU. MusicGen is autoregressive, so CPU generation is painfully slow.

## Install

We just need `transformers`.

In [ ]:
!pip install -q transformers

## The easy button — a music pipeline

Same `pipeline`, a new task: `text-to-audio`, pointed at the smallest MusicGen checkpoint. The prompt is the whole interface — be concrete about genre, instrument, mood, and tempo.

In [ ]:
from transformers import pipeline
import numpy as np

music = pipeline("text-to-audio", model="facebook/musicgen-small")
out = music("lo-fi hip hop with a mellow piano and vinyl crackle")
print(out["sampling_rate"])                    # 32000

Play it. `out["audio"]` is the waveform, `out["sampling_rate"]` is 32000 (MusicGen runs at 32 kHz). `np.squeeze` drops the extra length-1 dimension so `Audio` gets a flat array.

In [ ]:
from IPython.display import Audio

Audio(np.squeeze(out["audio"]), rate=out["sampling_rate"])

## The honest way — processor, model, generate

The processor turns your prompt into conditioning tensors; the model is the generator. `.to(device)` puts both on the GPU.

In [ ]:
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)

inputs = processor(
    text=["upbeat 8-bit chiptune for a boss fight"],
    padding=True,
    return_tensors="pt",
).to(device)

audio = model.generate(**inputs, max_new_tokens=256)   # ~5 s (≈50 tokens/sec)

That `model.generate(...)` is the **same autoregressive loop as a text model** — except the tokens are Encodec audio codes (Part 5), not words. `max_new_tokens` is literally how many audio tokens to predict, and MusicGen runs at ~50 tokens/sec, so 256 ≈ 5 seconds. **Token count is clip length.**

Play it. Fetch the sample rate from the model config (it's a property of the Encodec codec): mono, 32 kHz.

In [ ]:
from IPython.display import Audio

sr = model.config.audio_encoder.sampling_rate          # 32000
Audio(audio[0].cpu().numpy(), rate=sr)

## Try prompts that are specific

Stack genre + instrument + mood + tempo to steer hard. Naming what you *don't* want ("no drums") works too. Run each through `music(prompt)` (pipeline) or feed the whole list to the processor at once (by hand).

In [ ]:
prompts = [
    "epic orchestral film score, soaring strings and timpani",
    "warm bossa nova with nylon guitar and brushed drums",
    "dark ambient drone, slow and cinematic, no drums",
]

for p in prompts:
    out = music(p)
    print(p)
    display(Audio(np.squeeze(out["audio"]), rate=out["sampling_rate"]))

## Recap

MusicGen is a transformer doing next-token prediction over the Encodec vocabulary: **text → conditioning → predicted audio tokens → decoded waveform.** `musicgen-small` is the place to start; `-medium`/`-large` sound better but cost speed and memory. Output is mono at 32 kHz. Next up (Part 7): turning the dials — length, melody, and sampling.